# Model Training, Evaluation, and Explainability (Grad-CAM / Score-CAM)

This notebook orchestrates training/testing of a PyTorch image classification
model and applies class activation mapping (CAM-family) methods for
explainability.

**Note:** The core model architecture and training/testing loops
(`models.py`, `train.py`, `test.py`, `config_train.py`, `config_test.py`,
`utils.py`, `selfonn.py`) are part of a separate internal project and are
not included in this repository. This notebook shows the orchestration,
preprocessing, and explainability (XAI) pipeline only.

Update all file paths (marked with `<...>`) before running.


## Install or Update Libraries

In [ ]:
# !pip install torch==1.11.0 torchvision==0.12.0 torchaudio
# !pip install torchvision
# !pip install torchsummary
# !pip install torch torchvision
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install scikit-image
!pip install ipython
!pip install Pillow
!pip install matplotlib
!pip install seaborn
!pip install timm
!pip install opencv-python

In [ ]:

# Printing out all outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
# PyTorch
import torch
import torch.nn as nn
from torch import optim, cuda, tensor
from torch.utils.data import DataLoader
from torchvision import transforms, models
# warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
# Data science tools
import os
import timm
import numpy as np
from os import path
from importlib import import_module
# Visualizations
import cv2
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 14

# **Check GPU Status**
Check the available GPU from the Google Server, GPU model and other related information.

**Please MAKE sure that you are using GPU . Go to Runtime>> Change Runtime type  and select GPU**

In [ ]:
import torch
print("Is CUDA enabled GPU Available?", torch.cuda.is_available())
print("Is GPU Initialized yet?", torch.cuda.is_initialized())
print("GPU Number:", torch.cuda.device_count())
print("Current GPU Index:", torch.cuda.current_device())
print("GPU Type:", torch.cuda.get_device_name(device=None))
print("GPU Capability:", torch.cuda.get_device_capability(device=None))

In [ ]:
# NOTE: The following imports reference internal project files
# (config_test.py, config_train.py, selfonn.py, selfonn_custom_blocks.py,
# test.py, train.py, utils.py, models.py) which are not included here
# due to project confidentiality. Replace with your own model/training
# pipeline, or restore these files locally if you have access to them.
#
# from config_test import *
# from config_train import *
# from selfonn import *
# from selfonn_custom_blocks import *
# from test import *
# from train import *
# from utils import *
# from models import *


## **Train**

Training Configurations

Start Training

In [ ]:
# for Grive
# %cd /content/

# !python train.py
%run -m train

## **Test and Evaluate**







Copy the training results back to GDrive to acesss them later


---

Please note:

1.   **Results are going to be lost when you trun off the current section (Close the tab or leave it for more then 90 mins without any activity)**
2.   **Make sure that "Results" file is not in your directory to avoid overwriting any existing results**

Test Configurations

Evaluate the Test set

In [ ]:
!python test.py
# %run test.py # it will show the live output stream in this notebook

In [ ]:
# Check scikit-learn version
# pip show scikit-learn

# Upgrade scikit-learn if necessary
# pip install --upgrade scikit-learn


# **CAM**

Import custom functions

In [ ]:
%matplotlib inline
from CAM.cam_functions import CAM, GradCAM, GradCAMpp, SmoothGradCAMpp, ScoreCAM
from CAM.visualize import visualize, reverse_normalize
from CAM.imagenet_labels import label2idx, idx2label
from matplotlib import pyplot as plt

Import and process image

In [ ]:

normalize = transforms.Normalize(mean=[0.728 , 0.57346667, 0.77233333]  ,
                                 std= [0.1189, 0.1485, 0.0953] 
                                 )

preprocess = transforms.Compose([
    transforms.ToTensor(),
    normalize
    ])

Import Model and Extract Target Layer

In [ ]:
model_path = '<MODEL_CHECKPOINT_PATH>'
checkpoint = torch.load(model_path)
model = checkpoint['model']
model.eval()

In [ ]:
# Access the DenseNet model's last dense block
children = list(model.children())
features = children[0]
target_layer = list(features.children())[-2]  # Final dense block

print("Target layer for Grad-CAM:", target_layer)


In [ ]:
# target_layer = model.fc
# target_layer = target_layer[26]
# Choose the last layer

# children = list(model.children())
# target_layer = children[1] #select the final conv bloack as the target layer
# print(target_layer)

Load image and Create Warapper Model for CAM

In [ ]:
#This code is to run in colab only

# from google.colab.patches import cv2_imshow

# image_path = '/content/Data/Test/fold_1/Abnormal/train_174_Q1_n6.png'
# image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
# image = cv2.resize(image, (224,224), interpolation = cv2.INTER_CUBIC)  # Resize image
# cv2_imshow(image)
# image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
# tensor = preprocess(image) # convert image to tensor
# tensor = tensor.unsqueeze(0) # reshape 4D tensor (N, C, H, W)
# print(tensor.size())

In [ ]:
import cv2
import matplotlib.pyplot as plt

image_path = '<SAMPLE_IMAGE_PATH>'
image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_CUBIC)  # Resize image
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert BGR to RGB

# Display the image using matplotlib
plt.imshow(image)  
plt.axis('off')  # Turn off axis
plt.show()

tensor = preprocess(image)  # Convert image to tensor
tensor = tensor.unsqueeze(0)  # Reshape to 4D tensor (N, C, H, W)
print(tensor.size())


In [ ]:
# Configuration - Choose CAM Type
# Current Options: 'CAM', 'GradCAM', 'GradCAM++', 'SmoothGradCAM++', 'ScoreCAM
wrapper_model_type = 'ScoreCAM'
tensor = tensor.to('cpu')
model = model.to('cpu')

In [ ]:
wrapper_model = []
if wrapper_model_type == 'CAM':
    wrapper_model = CAM(model, target_layer)
elif wrapper_model_type == 'GradCAM':
    wrapper_model = GradCAM(model, target_layer)
elif wrapper_model_type == 'GradCAM++':
    wrapper_model = GradCAMpp(model, target_layer)
elif wrapper_model_type == 'SmoothGradCAM++':
    wrapper_model = SmoothGradCAMpp(model, target_layer, n_samples=25, stdev_spread=0.15)
elif wrapper_model_type == 'ScoreCAM':
    tensor = tensor.to('cuda')
    model = model.to('cuda')
    wrapper_model = ScoreCAM(model, target_layer)

Plot and Save CAM Heatmap

In [ ]:
cam, idx = wrapper_model(tensor)
print(idx2label[idx])
plt.imshow(cam.squeeze().numpy(), alpha=0.5, cmap='jet') # Visualize only cam

In [ ]:
#saving the heatmap
from PIL import Image
tensor = tensor.to('cpu')
img = reverse_normalize(tensor)
heatmap = np.swapaxes(np.swapaxes(np.squeeze(np.array(visualize(img, cam))), 0, 2), 0, 1)
heatmap = Image.fromarray((heatmap*255).astype(np.uint8))
heatmap.save('<HEATMAP_OUTPUT_PATH>')
plt.imshow(heatmap)